In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from prefect import flow

load_dotenv()

ROOT_DIR = Path.cwd().parent
ENV = os.getenv("ENV", default="dev")
os.chdir(ROOT_DIR)

QUERY = "How does the Transformer model use self-attention?"

print(f"Current Working Directory: {ROOT_DIR}")
print(f"Environment: {ENV}")
print(f"Query: {QUERY}")

Current Working Directory: c:\Users\nikhi\Documents\Projects\HybridRAG-Bench
Environment: dev
Query: How does the Transformer model use self-attention?


# Retrieve

In [2]:
from langchain_core.documents import Document
from src.generation_steps import retrieve


@flow(name="test-retrieve", log_prints=True)
def test_retrieve(query: str, env: str = "dev") -> list[Document]:
    """Thin wrapper flow so the Prefect task has a flow context."""
    return retrieve(query=query, env=env)


documents = test_retrieve(query=QUERY, env=ENV)
print(f"\nRetrieved {len(documents)} document(s)")

c:\Users\nikhi\Documents\Projects\HybridRAG-Bench\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


00:06:08.038 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8263
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

00:06:25.706 | INFO    | Flow run 'meteoric-owl' - Beginning flow run 'meteoric-owl' for flow 'test-retrieve'

00:06:26.403 | INFO    | Task run 'retrieve-ede' - Querying 'arxiv-research-rag-dev' (top_k=5, alpha=0.5)

00:06:44.029 | INFO    | Task run 'retrieve-ede' - Retrieved 5 document(s)

00:06:44.297 | INFO    | Task run 'retrieve-ede' - Finished in state Completed()

00:06:44.391 | INFO    | Flow run 'meteoric-owl' - Finished in state Completed()


Retrieved 5 document(s)


In [3]:
for i, doc in enumerate(documents[:3]):
    print(f"\n{'='*60}")
    print(f"Document {i}")
    print(f"{'='*60}")
    print(f"page_content[:200]: {doc.page_content[:200]}...")
    print(f"\nMetadata:")
    for key, value in doc.metadata.items():
        if key in ("text", "context_prefix"):
            print(f"  {key}: {str(value)[:100]}...")
        else:
            print(f"  {key}: {value}")

if len(documents) > 3:
    print(f"\n... and {len(documents) - 3} more document(s)")


Document 0
page_content[:200]: 3.2.3 Applications of Attention in our Model

The Transformer uses multi-head attention in three different ways:

In "encoder-decoder attention" layers, the queries come from the previous decoder laye...

Metadata:
  chunk_index: 7
  context_prefix: The document "Attention Is All You Need" introduces the Transformer model. This chunk details the th...
  has_table: False
  page_number: 5
  paper_title: Attention Is All You Need
  section: 3.2.3 Applications of Attention in our Model
  source_filename: Attention-Is-All-You-Need_170603762v7-9ab76e8d.pdf
  score: 0.580606818

Document 1
page_content[:200]: 3 Model Architecture

Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35]. Here, the encoder maps an input sequence of symbol representations (x1,...,xn) ...

Metadata:
  chunk_index: 2
  context_prefix: This chunk, from the 2017 Google Brain paper "Attention Is All You Need," introduces the Transformer...
  ha

# Format Context

In [4]:
from src.generation_steps import format_context


@flow(name="test-format-context", log_prints=True)
def test_format_context(documents: list[Document]) -> str:
    """Thin wrapper flow so the Prefect task has a flow context."""
    return format_context(documents=documents)


context = test_format_context(documents=documents)
print(f"\nContext length: {len(context)} chars")

00:06:45.509 | INFO    | Flow run 'coral-tench' - Beginning flow run 'coral-tench' for flow 'test-format-context'

00:06:45.530 | INFO    | Task run 'format_context-81f' - Formatted 5 source(s) into context (7218 chars)

00:06:45.544 | INFO    | Task run 'format_context-81f' - Finished in state Completed()

00:06:45.686 | INFO    | Flow run 'coral-tench' - Finished in state Completed()


Context length: 7218 chars


In [5]:
print(context)

[Source 1]
Paper: Attention Is All You Need
Section: 3.2.3 Applications of Attention in our Model
Page: 5

3.2.3 Applications of Attention in our Model

The Transformer uses multi-head attention in three different ways:

In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [38, 2, 9].

The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.

Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder

# Generate Answer

In [6]:
from src.generation_steps import generate_answer


@flow(name="test-generate-answer", log_prints=True)
def test_generate_answer(query: str, context: str, documents: list[Document]) -> dict:
    """Thin wrapper flow so the Prefect task has a flow context."""
    return generate_answer(query=query, context=context, documents=documents)


result = test_generate_answer(query=QUERY, context=context, documents=documents)
print(f"\nAnswer length: {len(result['answer'])} chars")
print(f"Sources: {len(result['sources'])}")

00:06:46.621 | INFO    | Flow run 'interesting-dugong' - Beginning flow run 'interesting-dugong' for flow 'test-generate-answer'

00:06:52.855 | INFO    | Task run 'generate_answer-0b5' - Token usage — input: 1699, output: 254

00:06:52.891 | INFO    | Task run 'generate_answer-0b5' - Generated answer (1327 chars) with 5 source(s)

00:06:52.900 | INFO    | Task run 'generate_answer-0b5' - Finished in state Completed()

00:06:52.993 | INFO    | Flow run 'interesting-dugong' - Finished in state Completed()


Answer length: 1327 chars
Sources: 5


In [7]:
print(result["answer"])

print(f"\n{'='*60}")
print("Sources:")
print(f"{'='*60}")
for src in result["sources"]:
    print(
        f"  [{src['source_number']}] {src['paper_title']} "
        f"| Section: {src.get('section', '—')} "
        f"| Page: {src.get('page_number', '—')}"
    )

The Transformer model uses self-attention in several ways:

1. **Encoder Self-Attention**: In the encoder, self-attention layers allow each position to attend to all positions in the previous layer of the encoder. This means that all keys, values, and queries come from the same source, specifically the output of the previous encoder layer [Source 1].

2. **Decoder Self-Attention**: Similarly, the decoder also contains self-attention layers that enable each position in the decoder to attend to all positions in the decoder up to and including that position. This is crucial for maintaining the auto-regressive property of the model, which prevents leftward information flow. To achieve this, the model masks out illegal connections in the input to the softmax function [Source 1].

3. **Multi-Head Attention**: The Transformer employs multi-head attention mechanisms in both the encoder and decoder. This allows the model to jointly attend to information from different representation subspaces a